# Scalable processing with xarray and Dask

This notebook demonstrates how xarray integrates with Dask to enable
lazy, chunked, and memory-aware computation on large climate datasets.


In [3]:
import matplotlib.pyplot as plt # type: ignore
import numpy as np # type: ignore
import xarray as xr # type: ignore
import dask # type: ignore
import dask.array as da # type: ignore

xr.set_options(keep_attrs=True, display_expand_data=False)
np.set_printoptions(threshold=10, edgeitems=2)

%xmode minimal
%matplotlib inline
%config InlineBackend.figure_format='retina'

Exception reporting mode: Minimal


#### Start a local Dask cluster

In [ ]:
from dask.distributed import Client # type: ignore

client = Client()
client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 11
Total threads: 11,Total memory: 18.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:51084,Workers: 11
Dashboard: http://127.0.0.1:8787/status,Total threads: 11
Started: Just now,Total memory: 18.00 GiB
Comm: tcp://127.0.0.1:51109,Total threads: 1
Dashboard: http://127.0.0.1:51110/status,Memory: 1.64 GiB
Nanny: tcp://127.0.0.1:51087,


We use a local Dask cluster to parallelize computations across CPU cores.
The same code can later run on HPC or distributed systems.


In [ ]:
# open the dataset as a dask array
ds = xr.open_dataset("../dataset/era5_temp.nc", chunks={'time': 'auto'
                                                        #, 
                                                        #"latitude": 2, 
                                                        #"longitude": 2
                                                        })
ds 

<xarray.Dataset> Size: 36GB
Dimensions:     (valid_time: 8760, latitude: 721, longitude: 1440)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 70kB 2025-01-01 ... 2025-12-31T23...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    expver      (valid_time) <U4 140kB dask.array<chunksize=(8760,), meta=np.ndarray>
Data variables:
    t2m         (valid_time, latitude, longitude) float32 36GB dask.array<chunksize=(674, 52, 103), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-06T10:57 GRIB to CDM+CF via cfgrib-0.9.1...

##### Inspect chunk sizes

In [6]:
ds.t2m.chunks

((674, 674, 674, 674, 674, 674, 674, 674, 674, 674, 674, 674, 672),
 (52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 45),
 (103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 101))

In [8]:
display("Representative chunk shape:", tuple(c[0] for c in ds.t2m.chunks))


'Representative chunk shape:'

(674, 52, 103)

#### Show the Dask-backed array

In [9]:
ds.t2m.data

dask.array<open_dataset-t2m, shape=(8760, 721, 1440), dtype=float32, chunksize=(674, 52, 103), chunktype=numpy.ndarray>

The variable is now backed by a Dask array.
This representation describes the computation graph, not the actual values.


#### Memory-aware computation with Dask

In [12]:
mean_t2m = ds.t2m.mean(dim="valid_time")
mean_t2m

<xarray.DataArray 't2m' (latitude: 721, longitude: 1440)> Size: 4MB
dask.array<chunksize=(52, 103), meta=np.ndarray>
Coordinates:
    number     int64 8B ...
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Attributes: (12/32)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown
    GRIB_surface:                             0.0

In [13]:
mean_t2m.compute()

<xarray.DataArray 't2m' (latitude: 721, longitude: 1440)> Size: 4MB
262.3 262.3 262.3 262.3 262.3 262.3 ... 229.1 229.1 229.1 229.1 229.1 229.1
Coordinates:
    number     int64 8B 0
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Attributes: (12/32)
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    long_name:                                2 metre temperature
    units:                                    K
    standard_name:                            unknown
    GRIB_surface:                             0.0